# BoW vs TF-IDF — Emotion Detection

Compare **Bag of Words (BoW)** and **TF-IDF** using five classifiers, with experiment tracking in **MLflow on DagsHub**.

This notebook filters the `happiness` and `sadness` classes and runs 10 experiments (5 algorithms × 2 vectorizers) as nested MLflow runs.

In [1]:
import re
import string
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
nltk.download('stopwords')
nltk.download('wordnet')

c:\Users\gupta\OneDrive\Desktop\MLOPS\emotion-detection-mlops\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gupta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gupta\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## 1. Configure DagsHub and MLflow

In [11]:
dagshub.init(
    repo_owner='Cheeku29',
    repo_name='emotion-detection-mlops',
    mlflow=True
)

print('Tracking URI:', mlflow.get_tracking_uri())
mlflow.set_experiment("BoW vs TF-IDF v2")

Initialized MLflow to track repo "Cheeku29/emotion-detection-mlops"

Repository Cheeku29/emotion-detection-mlops initialized!

Tracking URI: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow


2026/10/25 19:29:26 INFO mlflow.tracking.fluent: Experiment with name 'BoW vs TF-IDF v2' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/4558b1207b044e48a4a281902a05f08c', creation_time=1792936767411, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1792936767411, lifecycle_stage='active', name='BoW vs TF-IDF v2', tags={}, trace_location=None, workspace='default'>

## 2. Load the dataset

In [3]:
data_url = (
    'https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv'
)
df = pd.read_csv(data_url).drop(columns=['tweet_id'])
display(df.head())
print('Original dataset shape:', df.shape)
print(df['sentiment'].value_counts())

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


Original dataset shape: (40000, 2)
sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64


## 3. Clean and filter the text

In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\\S+|www\\.\\S+', ' ', text)
    text = re.sub(r'\\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\\s+', ' ', text).strip()
    words = [
        lemmatizer.lemmatize(word)
        for word in text.split()
        if word not in stop_words
    ]
    return ' '.join(words)


df['content'] = df['content'].apply(normalize_text)
df = df[df['sentiment'].isin(['happiness', 'sadness'])].copy()
df['sentiment'] = df['sentiment'].map({'sadness': 0, 'happiness': 1})
df = df.dropna(subset=['content', 'sentiment'])

X = df['content']
y = df['sentiment'].astype(int)

print('Filtered dataset shape:', df.shape)
print('Class distribution:')
display(y.value_counts().rename(index={0: 'sadness', 1: 'happiness'}))

Filtered dataset shape: (10374, 2)
Class distribution:


sentiment
happiness    5209
sadness      5165
Name: count, dtype: int64

## 4. Train/test split

Split the raw text first. Each vectorizer is fitted only on training text to prevent vocabulary/IDF leakage.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Training examples:', len(X_train))
print('Test examples:', len(X_test))

Training examples: 8299
Test examples: 2075


## 5. Define vectorizers and classifiers

In [8]:
vectorizers = {
    'BoW': CountVectorizer(max_features=3000, dtype=np.float32),
    'TF-IDF': TfidfVectorizer(max_features=3000, dtype=np.float32),
}


def get_models():
    # Fresh model instances for every experiment.
    return {
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
        'MultinomialNB': MultinomialNB(),
        'XGBoost': XGBClassifier(
            n_estimators=100, learning_rate=0.1, max_depth=3,
            random_state=42, eval_metric='logloss', n_jobs=-1
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=30,
            max_depth=20,
            max_features='sqrt',
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ),
        'GradientBoosting': GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
        ),
    }

## 6. Run experiments and log them to MLflow

The parent run groups all experiments. Each algorithm/vectorizer combination is a nested child run. `GradientBoostingClassifier` requires dense features, which can consume extra memory; if needed, replace it with `LinearSVC`.

In [12]:
results = []

with mlflow.start_run(run_name='BoW vs TF-IDF - All Experiments'):
    mlflow.log_params({
        'test_size': 0.2,
        'random_state': 42,
        'max_features': 3000,
        'classes': 'sadness, happiness',
    })

    for vec_name, vectorizer in vectorizers.items():
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        for algo_name, model in get_models().items():
            with mlflow.start_run(run_name=f'{algo_name} with {vec_name}', nested=True):
                if algo_name == 'GradientBoosting':
                    X_train_model = X_train_vec.toarray()
                    X_test_model = X_test_vec.toarray()
                else:
                    X_train_model = X_train_vec
                    X_test_model = X_test_vec

                model.fit(X_train_model, y_train)
                y_pred = model.predict(X_test_model)

                metrics = {
                    'accuracy': accuracy_score(y_test, y_pred),
                    'precision': precision_score(y_test, y_pred, zero_division=0),
                    'recall': recall_score(y_test, y_pred, zero_division=0),
                    'f1_score': f1_score(y_test, y_pred, zero_division=0),
                }

                mlflow.log_param('vectorizer', vec_name)
                mlflow.log_param('algorithm', algo_name)
                mlflow.log_params(model.get_params())
                mlflow.log_metrics(metrics)
                mlflow.sklearn.log_model(
                    model, name='model', serialization_format='pickle'
                )

                results.append({
                    'Algorithm': algo_name,
                    'Vectorizer': vec_name,
                    'Accuracy': metrics['accuracy'],
                    'Precision': metrics['precision'],
                    'Recall': metrics['recall'],
                    'F1 Score': metrics['f1_score'],
                })

                print(
                    f'{algo_name} with {vec_name} | '
                    f"Accuracy: {metrics['accuracy']:.4f} | "
                    f"Precision: {metrics['precision']:.4f} | "
                    f"Recall: {metrics['recall']:.4f} | "
                    f"F1: {metrics['f1_score']:.4f}"
                )

2026/10/25 19:30:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/25 19:30:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


LogisticRegression with BoW | Accuracy: 0.7889 | Precision: 0.7876 | Recall: 0.7937 | F1: 0.7906
🏃 View run LogisticRegression with BoW at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2/runs/6a472580fa244a679d5f082cf7b3769f
🧪 View experiment at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2


2026/10/25 19:30:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/25 19:30:41 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


MultinomialNB with BoW | Accuracy: 0.7812 | Precision: 0.7854 | Recall: 0.7764 | F1: 0.7809
🏃 View run MultinomialNB with BoW at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2/runs/34a05332fda94e8599ea1495cf62b37b
🧪 View experiment at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2


2026/10/25 19:30:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/25 19:31:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


...

XGBoost with TF-IDF | Accuracy: 0.7036 | Precision: 0.8280 | Recall: 0.5173 | F1: 0.6367
🏃 View run XGBoost with TF-IDF at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2/runs/b64a552d7717425eaa903173b6e036c1
🧪 View experiment at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2


2026/10/25 19:37:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/25 19:37:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RandomForest with TF-IDF | Accuracy: 0.7311 | Precision: 0.8040 | Recall: 0.6142 | F1: 0.6964
🏃 View run RandomForest with TF-IDF at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2/runs/837c0410d8064b41835cd7dc8cc62038
🧪 View experiment at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2


2026/10/25 19:41:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/25 19:42:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


GradientBoosting with TF-IDF | Accuracy: 0.7340 | Precision: 0.6879 | Recall: 0.8608 | F1: 0.7647
🏃 View run GradientBoosting with TF-IDF at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2/runs/dafe21865844422d97335d47c78e9a7a
🧪 View experiment at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2
🏃 View run BoW vs TF-IDF - All Experiments at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2/runs/43ab5715c5cf4fdeab126e6e780dffe9
🧪 View experiment at: https://dagshub.com/Cheeku29/emotion-detection-mlops.mlflow/#/experiments/2


## 7. Compare the results

In [13]:
results_df = pd.DataFrame(results).sort_values(
    by='F1 Score', ascending=False
).reset_index(drop=True)
display(results_df)

print('Best experiment by F1 Score:')
display(results_df.head(1))

,Algorithm,Vectorizer,Accuracy,Precision,Recall,F1 Score
0,LogisticRegression,BoW,0.788916,0.787619,0.793666,0.790631
1,LogisticRegression,TF-IDF,0.790843,0.795146,0.785988,0.790541
2,MultinomialNB,TF-IDF,0.782169,0.786965,0.776392,0.781643
3,MultinomialNB,BoW,0.781205,0.785437,0.776392,0.780888
4,GradientBoosting,TF-IDF,0.733976,0.687883,0.860845,0.764706
5,XGBoost,BoW,0.727711,0.681922,0.857965,0.759881
6,RandomForest,BoW,0.744578,0.801176,0.653551,0.719873
7,RandomForest,TF-IDF,0.731084,0.804020,0.614203,0.696409
8,GradientBoosting,BoW,0.708916,0.824926,0.533589,0.648019
9,XGBoost,TF-IDF,0.703614,0.827957,0.517274,0.636740


Best experiment by F1 Score:


,Algorithm,Vectorizer,Accuracy,Precision,Recall,F1 Score
0,LogisticRegression,BoW,0.788916,0.787619,0.793666,0.790631


## Notes

- Choose the best model based on the metric that matters for your task; F1 is a useful starting point for this binary classification experiment.
- MLflow logs the classifier, not the fitted vectorizer. For deployment, save the vectorizer too or log a complete scikit-learn `Pipeline`.
- Pickle artifacts should only be loaded from trusted sources.
- If DagsHub authentication is requested, complete the supported authentication flow; do not hardcode an access token in this notebook.